# 🎓 Advanced Prompt Engineering with OpenAI GPT-5
In this hands-on notebook, you will apply and experiment with advanced prompt engineering techniques for GPT-5, following OpenAI's latest best practices for agentic workflows, coding, reasoning, and steerability.

# 1️⃣ Setup: OpenAI Client
We’ll load credentials from a .env file and initialize the OpenAI Python SDK for GPT-5.

In [12]:
# 1️⃣ Setup cell: Import libraries and load environment variables  
  
import os  
from openai import AzureOpenAI  
from dotenv import load_dotenv  
  
# Load Azure credentials from a .env file (you should have credentials.env in your directory)  
load_dotenv("credentials.env")  
  
# Create Azure OpenAI client  
client = AzureOpenAI(  
    api_key=os.getenv("AZURE_OPENAI_KEY"),  
    api_version="2025-04-01-preview",  
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")  
)  
  
print("✅ Azure OpenAI client initialized.")  

✅ Azure OpenAI client initialized.


# 2️⃣ Basic Chat Completion
Let’s ensure our setup works by sending a simple prompt.

In [13]:
model_name = "gpt-5"  # Replace with your model/deployment name if needed  
  
response = client.chat.completions.create(  
    model=model_name,  
    messages=[  
        {"role": "system", "content": "You are a helpful assistant."},  
        {"role": "user", "content": "Hello! What can you do?"}  
    ]  
)  
print(response.choices[0].message.content)  

Hi! I can help with a wide range of text-based tasks. Here are some things I’m good at:

- Explain and teach: break down complex topics, walk through step-by-step solutions, create study guides and quizzes.
- Write and edit: emails, resumes, reports, cover letters, blog posts, stories, scripts; improve clarity, tone, and grammar.
- Brainstorm and create: ideas for projects, content outlines, names, taglines, marketing copy, prompts.
- Plan and organize: study plans, workout routines, travel itineraries, event checklists, project timelines.
- Solve problems and math: show work, check your reasoning, help with proofs and quantitative questions.
- Code help: write, explain, and debug code; review logic; suggest tests; translate between languages.
- Data/text analysis: summarize long documents, extract key points, compare options, make tables or checklists.
- Translate and localize: between many major languages; adjust tone and register.

What I can’t do by default: browse the live web or 

# 3️⃣ Controlling Agentic Eagerness
GPT-5’s agentic behavior can be calibrated via both prompts and the reasoning_effort parameter.

3.1 Prompting for Less Eagerness
To make GPT-5 less proactive and more efficient, specify clear limits and escape hatches.

In [14]:
system_prompt = """  
You are an assistant in an agentic workflow.  
<context_gathering>  
- Goal: Get enough context fast. Start broad, then focus.  
- Run parallel, targeted queries; deduplicate paths; avoid over-searching.  
- Early stop criteria: when you can name exact content to change, or 70% of top hits converge.  
- Search depth: very low. Max 2 tool calls.  
- If you need more investigation, update the user and ask for confirmation.  
- Proceed even if not fully certain, if needed.  
</context_gathering>  
"""  
  
response = client.chat.completions.create(  
    model=model_name,  
    messages=[  
        {"role": "system", "content": system_prompt},  
        {"role": "user", "content": "Can you summarize the main changes in this codebase?"}  
    ],  
    reasoning_effort="low"  # Lower exploration  
)  
print(response.choices[0].message.content)  

I don’t have the code or a link yet. To summarize the main changes, I need a comparison point and access to the repo. Could you provide one of the following?

- Repo URL (GitHub/GitLab/etc.) and which branch to analyze
- The two points to compare (e.g., tag v1.3.0 to main, or commit A to commit B)
- Or upload a diff/patch or a recent git log and diffstat

If you prefer to run a couple commands and paste the output, this gives me enough to produce a solid summary:
- High-level history:
  git log --oneline --decorate --graph --no-merges -n 200
- Change size by file:
  git diff --stat <BASE>..<HEAD>
- Notable files touched most:
  git log --name-only --pretty=format: <BASE>..<HEAD> | sort | uniq -c | sort -nr | head -50
- Dependency/config changes:
  git diff <BASE>..<HEAD> -- package*.json yarn.lock pnpm-lock.yaml requirements*.txt pyproject.toml setup.cfg poetry.lock Cargo.toml Cargo.lock go.mod go.sum Gemfile* Podfile* build.gradle* pom.xml composer.json composer.lock
- Schema/migratio

3.2 Prompting for More Eagerness
To encourage persistence and full autonomy, use this style:

In [15]:
system_prompt = """  
<persistence>  
- You are an agent. Keep going until the user's query is completely resolved.  
- Only end your turn when the problem is solved.  
- Never stop at uncertainty — research or deduce the most reasonable approach and continue.  
- Do not ask the human to confirm assumptions — act and document them for the user.  
</persistence>  
"""  
  
response = client.chat.completions.create(  
    model=model_name,  
    messages=[  
        {"role": "system", "content": system_prompt},  
        {"role": "user", "content": "Help me debug this function. It’s returning None unexpectedly."}  
    ],  
    reasoning_effort="high"  
)  
print(response.choices[0].message.content)  

I’m going to assume Python (because you said “None”). Here’s a focused way to find and fix why your function is returning None.

Most common causes and fixes
- Some path doesn’t return: If any execution path reaches the end of a def without hitting return, Python returns None.
  Example:
    def sign(x):
        if x > 0: return 1
        elif x < 0: return -1
        # x == 0 falls through -> None
  Fix: Make all paths return, e.g. add else: return 0.
- Returning the result of an in-place method (these return None):
  List/set/dict/random in-place methods: list.sort, list.append, list.extend, list.insert, list.clear, set.add, set.update, set.clear, dict.update, random.shuffle, pathlib.Path.write_text? (returns int; fine) but many pandas “inplace=True” methods return None.
  Example:
    def sorted_copy(xs):
        return xs.sort()  # None
  Fix:
    return sorted(xs)  # functional
    # or mutate and then return the object explicitly
    xs.sort()
    return xs
- Pandas in-place oper

# 4️⃣ Tool Preambles & Progress Updates
Steer how GPT-5 narrates its plan and progress for tool-based (agentic) workflows.

In [16]:
system_prompt = """  
<tool_preambles>  
- Always begin by rephrasing the user's goal in a friendly, clear, and concise manner.  
- Immediately outline a structured plan with logical steps.  
- As you execute, narrate each step succinctly, marking progress clearly.  
- Summarize completed work distinctly from your upfront plan.  
</tool_preambles>  
"""  
  
response = client.chat.completions.create(  
    model=model_name,  
    messages=[  
        {"role": "system", "content": system_prompt},  
        {"role": "user", "content": "Check the weather in San Francisco and tell me in Celsius and Fahrenheit."}  
    ]  
)  
print(response.choices[0].message.content)  

Your goal: Get the current weather for San Francisco and report the temperature in both Celsius and Fahrenheit.

Plan:
1) Confirm the exact location (San Francisco, California, USA).
2) Access a live weather source to get the current conditions and temperature.
3) Present the temperature in both °C and °F (and optionally a brief forecast).
4) If you’d like, set up quick ways to check it again later.

Step 1 — Confirm and request access: I don’t have live web access by default. Would you like me to fetch the real-time weather now? Please confirm:
- Is the location San Francisco, California, USA?
- Do you want only the current temperature, or also current conditions and a short forecast?

If you prefer not to enable web fetching:
- Quick manual check: Visit weather.gov or weather.com and search “San Francisco, CA,” or run “curl wttr.in/San+Francisco” in a terminal. Share the temperature you see, and I’ll immediately convert and present it in both °C and °F.

Progress: Awaiting your go-ah

# 5️⃣ Reasoning Effort & Minimal Reasoning
The reasoning_effort parameter controls how much the model “thinks” and how persistently it calls tools.

Use "minimal" or "low" for latency-sensitive tasks.
Use "high" for complex, multi-step tasks.

In [17]:
response = client.chat.completions.create(  
    model=model_name,  
    messages=[  
        {"role": "system", "content": "You are a helpful assistant."},  
        {"role": "user", "content": "Give me a quick summary of the main points in this article:"}  
    ],  
    reasoning_effort="minimal"  
)  
print(response.choices[0].message.content)  

I don’t see the article content. Please paste the text or share the link, and I’ll give you a concise summary of the main points. If it’s long, let me know your preferred length (e.g., 3–5 bullet points or a one-paragraph summary).


# 6️⃣ Coding Prompts: Planning, Self-Reflection, and Style
6.1 Zero-to-One App Generation (Self-Reflection)
Encourage world-class results by instructing the model to plan and self-critique.

In [18]:
system_prompt = """  
<self_reflection>  
- First, think of a rubric for a world-class one-shot web app (5-7 categories).  
- Internally iterate on your solution using this rubric.  
- Only output your best result.  
</self_reflection>  
"""  
  
response = client.chat.completions.create(  
    model=model_name,  
    messages=[  
        {"role": "system", "content": system_prompt},  
        {"role": "user", "content": "Build a simple Next.js app that displays a list of recent news articles."}  
    ],  
    verbosity="high"  
)  
print(response.choices[0].message.content)  

TypeError: Completions.create() got an unexpected keyword argument 'verbosity'

6.2 Matching Codebase Design Standards
Provide project-specific engineering principles and directory structure to ensure the code “blends in.”

In [ ]:
system_prompt = """  
<code_editing_rules>  
<guiding_principles>  
- Clarity and Reuse: Components must be modular and reusable.  
- Consistency: Adhere to the design system for colors, typography, spacing.  
- Simplicity: Favor small, focused components.  
- Visual Quality: Match OSS guidelines for spacing, padding, hover states, etc.  
</guiding_principles>  
<frontend_stack_defaults>  
- Framework: Next.js (TypeScript)  
- Styling: TailwindCSS  
- UI Components: shadcn/ui  
- Icons: Lucide  
- State Management: Zustand  
- Directory Structure:  

/src
/app
/api//route.ts
/(pages)
/components/
/hooks/
/lib/
/stores/
/types/
/styles/

In [ ]:
</frontend_stack_defaults>  
<ui_ux_best_practices>  
- Limit font sizes for hierarchy; use semantic HTML; use 1 neutral and 2 accent colors; padding/margins in multiples of 4.  
</ui_ux_best_practices>  
</code_editing_rules>  
"""  
  
response = client.chat.completions.create(  
    model=model_name,  
    messages=[  
        {"role": "system", "content": system_prompt},  
        {"role": "user", "content": "Add a button component that matches the project’s style."}  
    ]  
)  
print(response.choices[0].message.content)  

# 7️⃣ Verbosity & Output Formatting
Control GPT-5’s answer length with the verbosity parameter and prompt overrides.

In [ ]:
response = client.chat.completions.create(  
    model=model_name,  
    messages=[  
        {"role": "system", "content": "You are a helpful assistant. Please answer concisely."},  
        {"role": "user", "content": "What are the key features of GPT-5?"}  
    ],  
    verbosity="low"  
)  
print(response.choices[0].message.content)  

To override globally-set verbosity for code outputs, include explicit instructions in your prompt:

In [ ]:
system_prompt = """  
- Use high verbosity when outputting code or code tool results.  
- For all other answers, keep text brief.  
"""  
  
response = client.chat.completions.create(  
    model=model_name,  
    messages=[  
        {"role": "system", "content": system_prompt},  
        {"role": "user", "content": "Write a function to reverse a linked list in Python."}  
    ],  
    verbosity="low"  
)  
print(response.choices[0].message.content)  

# 8️⃣ Markdown Formatting
By default, GPT-5 disables Markdown. To enforce Markdown where appropriate, instruct the model clearly.

In [ ]:
system_prompt = """  
- Use Markdown **only where semantically correct** (e.g., `inline code`, ```code fences```, lists, tables).  
- Use backticks for file, directory, function, and class names.  
"""  
  
response = client.chat.completions.create(  
    model=model_name,  
    messages=[  
        {"role": "system", "content": system_prompt},  
        {"role": "user", "content": "Show me an example of a Python function that returns the square of a number."}  
    ]  
)  
print(response.choices[0].message.content)  

# 9️⃣ Metaprompting: GPT-5 as a Prompt Optimizer
Ask GPT-5 to critique and optimize your own prompts!

In [ ]:
metaprompt = """  
When asked to optimize prompts, give answers from your own perspective — explain what specific phrases could be added to, or deleted from, this prompt to more consistently elicit the desired behavior or prevent the undesired behavior.  
  
Here's a prompt: [PROMPT]  
The desired behavior from this prompt is for the agent to [DO DESIRED BEHAVIOR], but instead it [DOES UNDESIRED BEHAVIOR]. While keeping as much of the existing prompt intact as possible, what are some minimal edits/additions that you would make to encourage the agent to more consistently address these shortcomings?  
"""  
  
response = client.chat.completions.create(  
    model=model_name,  
    messages=[  
        {"role": "system", "content": metaprompt},  
        {"role": "user", "content": "Here's a prompt: 'Summarize this article.' The desired behavior is a concise summary, but instead the agent gives a verbose, multi-page explanation."}  
    ]  
)  
print(response.choices[0].message.content)  

# 🔟 Instruction Following: Avoid Contradictory Prompts
GPT-5 is sensitive to vague or conflicting instructions. Always review your prompts for contradictions!

Bad Example (contradictory):

Never schedule an appointment without explicit patient consent recorded in the chart.  
Auto-assign the earliest same-day slot without contacting the patient as the first action to reduce risk.  

Better:
Never schedule an appointment without explicit patient consent recorded in the chart.  
Auto-assign the earliest same-day slot after informing the patient.  

# 🧪 Experiment: Try Your Own Prompt Engineering!
Use this cell as a playground for your own advanced prompts and API parameters.

In [ ]:
# Example: Try a custom prompt with agentic workflow, code editing, or planning!  
custom_system = """  
You are a code review assistant. For every code block you see, give a brief summary of its purpose, point out any security issues, and suggest improvements in Markdown.  
"""  
  
user_input = """  
def authenticate(user, password):  
    if password == "12345":  
        return True  
    return False  
"""  
  
response = client.chat.completions.create(  
    model=model_name,  
    messages=[  
        {"role": "system", "content": custom_system},  
        {"role": "user", "content": user_input}  
    ],  
    verbosity="medium"  
)  
print(response.choices[0].message.content)  

# ✅ Summary
- Prompt engineering for GPT-5 leverages both new API parameters (reasoning_effort, verbosity) and explicit, structured prompts.
- Agentic workflows can be calibrated for more or less autonomy.
- Tool preambles and progress narration improve user experience in agentic flows.
- Self-reflection, planning, and adherence to coding standards improve code quality.
- Avoid contradictions in your instructions for maximal performance.

- Tip: For more, see the official GPT-5 Prompting Guide: https://cookbook.openai.com/examples/gpt-5/gpt-5_prompting_guide

- Learn More: https://github.com/microsoft/poml?tab=readme-ov-file